In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import networkx as nx
import json

In [ ]:
# Load jsonl file into a list of dictionaries
with open("../data/labelled/all_parallel.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

In [ ]:
from util import extract_ner_labels

b = nx.Graph()

# Create ent-doc pairs
edges = []
for item in data:
    doc_id = (item["id"], "DOC")
    entities = [(e[0].title(), e[1]) for e in extract_ner_labels(item["text"])]

    for e in entities:
        edges.append((e, doc_id))

b.add_edges_from(edges)

print(f"Nodes: {len(b.nodes())}\nEdges: {len(b.edges())}")
print(f"Is bipartite: {nx.is_bipartite(b)}")

In [ ]:
from networkx.algorithms.minors import contracted_nodes

from Levenshtein import distance
import numpy as np


nodes = list(b.nodes())
sim_matrix = np.full((len(nodes), len(nodes)), np.inf)

for i in range(len(nodes)):
    for j in range(i, len(nodes)):
        if (isinstance(nodes[i][0], int)) or (isinstance(nodes[j][0], int)):
            continue
        else:
            sim_matrix[i, j] = distance(nodes[i][0], nodes[j][0])

m = np.matrix(sim_matrix)

similar_ent = np.argwhere((m <= 1) & (m >= 0))

for i, j in similar_ent:
    if i == j:
        continue
    print(nodes[i], "|", nodes[j], "|", m[i, j])
    try:
        print(f"Degree for node i: {b.degree()[nodes[i]]}")
        print(f"Degree for node j: {b.degree()[nodes[j]]}")

        # Contract the two nodes into the node with the higher degree
        if b.degree()[nodes[i]] >= b.degree()[nodes[j]]:
            b = contracted_nodes(b, nodes[i], nodes[j])
        else:
            b = contracted_nodes(b, nodes[j], nodes[i])

    except KeyError:
        print(f"{nodes[i]} or {nodes[j]} already contracted")

In [ ]:
print(f"Nodes: {len(b.nodes())}\nEdges: {len(b.edges())}")
print(f"Is bipartite: {nx.is_bipartite(b)}")

In [ ]:
from util import draw_bipartite

draw_bipartite(b, draw_labels=False, figsize=(10, 5))

In [ ]:
from networkx.algorithms.bipartite.projection import projected_graph
from util import draw_graph

entity_nodes = [n for n in b.nodes() if not isinstance(n[0], int)]
p = projected_graph(b, entity_nodes)

draw_graph(p)

In [ ]:
ego = nx.ego_graph(p, ("Enrico Fontana Paolo Conti", "PER"))

draw_graph(
    ego,
    figsize=(7, 5),
    node_size=150,
    label_font_size=10,
)